# 0 Introducción

En este notebook trabajaremos con **bases de datos y tablas en Databricks**.

A lo largo de los siguientes pasos veremos cómo **crear y gestionar tablas**, así como cómo interactuar con ellas utilizando comandos SQL.

Para comenzar, vamos a **crear una tabla** que utilizaremos en los ejemplos del notebook.

# 1 Managed Tables

In [0]:
USE CATALOG workspacedemofull02;

USE SCHEMA default;

## 1.1 Creación de una tabla gestionada

Empecemos creando una tabla que llamaremos **`managed_default`** e introduciendo algunos datos en ella.

Esta será una **tabla gestionada (managed table)**, ya que **no estamos especificando la palabra clave `LOCATION`** al crearla.

En Databricks, cuando no se define una ubicación explícita, el sistema **gestiona automáticamente dónde se almacenan los datos de la tabla dentro del almacenamiento del metastore**.

In [0]:
CREATE TABLE managed_default
  (width INT, length INT, height INT);

INSERT INTO managed_default
VALUES (3 INT, 2 INT, 1 INT)

num_affected_rows,num_inserted_rows
1,1


## 1.2 Explorando los metadatos de la tabla

Ejecutemos el comando **`DESCRIBE EXTENDED`** sobre nuestra tabla para revisar su información de metadatos.

Si nos desplazamos hacia la parte inferior del resultado veremos **dos informaciones importantes** sobre la tabla.

La primera es la **ubicación (`Location`)**. Aquí podemos comprobar dónde se almacenan físicamente los datos de la tabla.  
En nuestro caso, la tabla se ha creado **bajo el metastore por defecto**. En la versión **Free** de Databricks, esto puede aparecer simplemente como un **espacio en blanco en el campo de ubicación**.

La segunda información relevante es el **tipo de tabla (`Type`)**, que aparece como **`MANAGED`**.  
Esto confirma que se trata de **una tabla gestionada**, ya que **no especificamos la palabra clave `LOCATION` durante su creación**.

<img src="https://raw.githubusercontent.com/jmartinezceste/Course_Delta_Lake/main/delta_img/delta_ses2_1.png" width="1600px"/>

In [0]:
DESCRIBE EXTENDED managed_default

col_name,data_type,comment
width,int,null
length,int,null
height,int,null
,,
# Delta Statistics Columns,,
Column Names,"width, length, height",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,workspacedemofull02,



# 2 External Tables

## 2.1 Creación de una tabla externa

Ahora vamos a crear una **tabla externa** en el mismo **schema `default`**.

Para crear una tabla externa basta con **especificar la palabra clave `LOCATION` en la sentencia `CREATE TABLE`**, seguida de la **ruta donde se almacenarán físicamente los datos de la tabla**.

En nuestro caso utilizaremos **el mismo bucket de S3 que empleamos en la sesión anterior**.

A continuación, **crearemos la tabla e insertaremos algunos datos** para poder trabajar con ella en los siguientes pasos.

In [0]:
%python
import re

# Get current user
user = spark.sql("SELECT current_user()").first()[0]

# Clean username
user_clean = re.sub(r"@.*", "", user)
user_clean = user_clean.replace(".", "_").replace("-", "_")

table_name = "external_default"
location = f"s3://mi-bucket-publico-javier-2026/tables/{user_clean}/{table_name}"

print(user_clean)
print(location)

test_data_jm
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/external_default


In [0]:
%python
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_name}
(
  width INT,
  length INT,
  height INT
)
LOCATION '{location}'
""")

spark.sql(f"""
INSERT INTO {table_name}
VALUES (3, 2, 1)
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

## 2.2 Verificando los metadatos de la tabla externa

Ejecutemos ahora el comando **`DESCRIBE EXTENDED`** sobre nuestra tabla externa.

En el resultado podremos observar dos aspectos clave:

- El campo **`Type`** indica **`EXTERNAL`**, lo que confirma que se trata de una **tabla externa**.
- El campo **`Location`** muestra **la ruta específica que definimos al crear la tabla**, es decir, el directorio en S3 donde se almacenan físicamente sus archivos de datos.

Esto confirma que los datos **no están gestionados por el metastore**, sino que **residen en la ubicación externa que hemos especificado**.

<img src="https://raw.githubusercontent.com/jmartinezceste/Course_Delta_Lake/main/delta_img/delta_ses2_2.png" width="1600px"/>

In [0]:
DESCRIBE EXTENDED external_default

col_name,data_type,comment
width,int,null
length,int,null
height,int,null
,,
# Delta Statistics Columns,,
Column Names,"width, length, height",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,workspacedemofull02,



# 3 Dropping Tables

## 3.1 Eliminación de una tabla gestionada

Veamos ahora qué ocurre si **eliminamos la tabla gestionada**.

Al ejecutar el comando `DROP TABLE` recibiremos un mensaje indicando que **la tabla ha sido eliminada correctamente**.

En el caso de una **tabla gestionada**, Databricks no solo elimina la definición de la tabla del catálogo, sino que también **elimina los archivos de datos asociados en el almacenamiento**.

In [0]:
DROP TABLE managed_default

## 3.2 Verificando la eliminación de los datos en Managed Table

Podemos confirmarlo **revisando el directorio donde se almacenaban los datos de la tabla gestionada**.

Al hacerlo veremos que **el directorio ya no existe o está vacío**, lo que confirma que **los archivos de datos también han sido eliminados junto con la tabla**.

In [0]:
%fs ls 'dbfs:/user/hive/warehouse/managed_default'

In [0]:
select * from managed_default

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8083441662026207>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'select * from managed_table\n')

File /databricks/python/lib/python3.11/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:152, in SqlMagic.sql(self, line, cell)
    148     raise Exception(
    149         "Cannot run %sql command because spark c

Y efectivamente, al revisar el almacenamiento podemos comprobar que **el directorio de la tabla y todos sus archivos han sido eliminados**.

Esto confirma el comportamiento de las **tablas gestionadas** en Databricks: cuando se elimina la tabla con `DROP TABLE`, **también se eliminan automáticamente los datos físicos asociados**.

## 3.3 Eliminación de la tabla externa

Eliminemos ahora **la tabla externa** utilizando el mismo comando `DROP TABLE`.

A diferencia de lo que ocurría con las **tablas gestionadas**, en este caso el comando **solo elimina la definición de la tabla del catálogo**, pero **no elimina los archivos de datos del almacenamiento**.

Veamos ahora qué ocurre en el almacenamiento tras ejecutar esta operación.

In [0]:
DROP TABLE external_default

## 3.4 Comportamiento al eliminar una tabla externa

Si volvemos al **Catalog**, podremos comprobar que **la tabla ya no aparece**, lo que confirma que su definición ha sido eliminada del metastore.

Sin embargo, si revisamos el **directorio físico donde se almacenaban los datos**, veremos que **el directorio de la tabla y sus archivos siguen existiendo**.

Esto ocurre porque la tabla fue creada **fuera del directorio gestionado de la base de datos**, utilizando una **ubicación externa**.  
En este caso, **los datos subyacentes no son gestionados por Unity Catalog**, por lo que **al eliminar la tabla solo se elimina la metadata**, pero **los archivos de datos permanecen en el almacenamiento**.

In [0]:
%python
display(dbutils.fs.ls(location))

path,name,size,modificationTime
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/external_default/part-00000-d2717016-5742-4762-a66b-a1e242f6fbe2.c000.snappy.parquet,part-00000-d2717016-5742-4762-a66b-a1e242f6fbe2.c000.snappy.parquet,977,1776097422000
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/external_default/_delta_log/,_delta_log/,0,1776097484636


# 4 Creación de nuevos schemas

Además del **schema `default`**, también podemos crear **schemas adicionales** para organizar mejor nuestras tablas.

En Databricks podemos utilizar tanto la sintaxis **`CREATE SCHEMA`** como **`CREATE DATABASE`**, ya que **ambos comandos son equivalentes** y crean el mismo tipo de objeto.

In [0]:
CREATE SCHEMA new_default

## 4.1 Explorando los metadatos del schema

Ejecutemos ahora el comando **`DESCRIBE EXTENDED`** sobre el schema para revisar su información de metadatos.

En el resultado podremos ver la **ubicación (`Location`)** donde se ha creado esta nueva base de datos dentro del almacenamiento.

Estamos en la version Free y el Location en los schemas managed aparecen vacios pero debríamos ver que la base de datos aparece con la **extensión `.db`**. Esta convención se utiliza para **diferenciar los directorios de bases de datos de otras carpetas que contienen tablas** dentro del mismo directorio de almacenamiento.

In [0]:
DESCRIBE DATABASE EXTENDED new_default

database_description_item,database_description_value
Catalog Name,workspacedemofull02
Namespace Name,new_default
Comment,
Collation,UTF8_BINARY
Location,
Owner,test.data.jm@gmail.com
Properties,
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_2)


## 4.2 Creación de tablas en la nueva base de datos

Ahora vamos a crear **algunas tablas dentro de esta nueva base de datos**.

En este ejemplo crearemos **dos tipos de tablas**:

- Una **tabla gestionada (managed table)**, donde Databricks se encargará de gestionar automáticamente la ubicación de los datos.
- Una **tabla externa (external table)**, donde especificaremos explícitamente la **ruta de almacenamiento** utilizando la palabra clave `LOCATION`.

Esto nos permitirá observar **las diferencias de comportamiento entre ambos tipos de tablas** dentro de un mismo schema.

In [0]:
USE new_default;

In [0]:
%python
import re

# Get current user
user = spark.sql("SELECT current_user()").first()[0]

# Clean username
user_clean = re.sub(r"@.*", "", user)
user_clean = user_clean.replace(".", "_").replace("-", "_")

table_name1 = "external_NEW_default"
table_name2 = "managed_NEW_default"
location = f"s3://mi-bucket-publico-javier-2026/tables/{user_clean}/{table_name2}"

print(user_clean)
print(location)

test_data_jm
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/managed_NEW_default


In [0]:
%python

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_name2}
(
  width INT,
  length INT,
  height INT
)
""")

spark.sql(f"""
INSERT INTO {table_name2}
VALUES (3, 2, 1)
""")

"-----------------------------------"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_name1}
(
  width INT,
  length INT,
  height INT
)
LOCATION '{location}'
""")

spark.sql(f"""
INSERT INTO {table_name1}
VALUES (3, 2, 1)
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

## 4.3 Metadatos de la tabla gestionada

Si ejecutamos el comando **`DESCRIBE EXTENDED`** sobre la tabla **managed**, podemos observar dos aspectos importantes en los metadatos.

En primer lugar, el campo **`Type`** indica que la tabla es de tipo **`MANAGED`**, lo que confirma que Databricks gestiona automáticamente tanto la metadata como los archivos de datos asociados.

Location aparece vacio pero deberiamos ver la ruta donde se almacenan físicamente los datos. En esta ruta aparece el directorio **`new_default.db`**, que corresponde al **schema que acabamos de crear**, lo que indica que la tabla se almacena dentro del directorio gestionado de esa base de datos.

In [0]:
DESCRIBE EXTENDED managed_new_default

col_name,data_type,comment
width,int,null
length,int,null
height,int,null
,,
# Delta Statistics Columns,,
Column Names,"width, length, height",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,workspacedemofull02,


## 4.4 Metadatos de la tabla externa

Si ejecutamos el comando **`DESCRIBE EXTENDED`** sobre la **tabla externa**, veremos que el campo **`Type`** aparece como **`EXTERNAL`**, lo que confirma que se trata de una **tabla externa**.

En el campo **`Location`** también podemos observar la **ruta donde se almacenan los datos**. En esta ruta aparece igualmente el directorio **`.db`**, que corresponde al **schema donde se ha registrado la tabla**, aunque los datos se encuentren en una ubicación definida explícitamente mediante `LOCATION`.

In [0]:
DESCRIBE EXTENDED external_new_default

col_name,data_type,comment
width,int,null
length,int,null
height,int,null
,,
# Delta Statistics Columns,,
Column Names,"width, length, height",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,workspacedemofull02,


## 4.5 Eliminación de las tablas

Podemos ahora **eliminar ambas tablas** utilizando el comando `DROP TABLE`.

Al hacerlo, veremos nuevamente que **el directorio de la tabla gestionada (`managed table`) y sus archivos de datos subyacentes han sido eliminados**, ya que Databricks gestiona automáticamente tanto la metadata como el almacenamiento de este tipo de tablas.

In [0]:
DROP TABLE managed_new_default;
DROP TABLE external_new_default;

In [0]:
%fs ls 'dbfs:/user/hive/warehouse/new_default.db/managed_new_default'

In [0]:
%python
display(dbutils.fs.ls(location))

path,name,size,modificationTime
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/managed_NEW_default/part-00000-a70e216a-2bb0-4081-a4e5-4917258caac9.c000.snappy.parquet,part-00000-a70e216a-2bb0-4081-a4e5-4917258caac9.c000.snappy.parquet,977,1776097660000
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/managed_NEW_default/_delta_log/,_delta_log/,0,1776097770675


# 5 Creación de una base de datos con ubicación específica

Por último, vamos a crear una **base de datos en una localización específica dentro de nuestro bucket público**.

Para ello, utilizaremos el comando `CREATE DATABASE` (o `CREATE SCHEMA`) indicando la **ruta en S3 mediante la palabra clave `LOCATION`**.

De esta forma, todas las **tablas gestionadas que creemos dentro de esta base de datos** se almacenarán directamente en esa **ubicación definida en el bucket**, en lugar de utilizar el almacenamiento por defecto del metastore.

In [0]:
%python
import re

# Get current user
user = spark.sql("SELECT current_user()").first()[0]

# Clean username
user_clean = re.sub(r"@.*", "", user)
user_clean = user_clean.replace(".", "_").replace("-", "_")

# table_name = "external_default"
location4 = f"s3://mi-bucket-publico-javier-2026/tables/{user_clean}/"

print(user_clean)
print(location4)

test_data_jm
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/


In [0]:
%python
spark.sql(f"""
          CREATE DATABASE IF NOT EXISTS custom 
          MANAGED LOCATION '{location4}'""")

DataFrame[]

## 5.1 Verificando la ubicación de la base de datos

La nueva base de datos aparecerá creada correctamente en **Unity Catalog**.

Sin embargo, si ejecutamos el comando **`DESCRIBE DATABASE EXTENDED`**, podremos comprobar que su **ubicación** corresponde a la **ruta personalizada que hemos definido durante la creación** de la base de datos.

Es decir, aunque la base de datos esté registrada en **Unity Catalog**, sus objetos se almacenarán en una **ubicación distinta del directorio gestionado por defecto de UC**.

In [0]:
DESCRIBE DATABASE EXTENDED custom

database_description_item,database_description_value
Catalog Name,workspacedemofull02
Namespace Name,custom
Comment,
Collation,UTF8_BINARY
Location,s3://mi-bucket-publico-javier-2026/tables/test_data_jm/__unitystorage/schemas/4deafdca-79e8-4558-9662-dcb6273e5f69
Owner,test.data.jm@gmail.com
RootLocation,s3://mi-bucket-publico-javier-2026/tables/test_data_jm
Properties,
Predictive Optimization,ENABLE (inherited from METASTORE metastore_aws_us_east_2)


## 5.2 Uso de la base de datos con ubicación personalizada

Esta base de datos **no tiene un comportamiento especial a nivel de uso**.  
Podemos crear tablas de forma completamente normal, tanto **tablas gestionadas (managed)** como **tablas externas (external)**.

Al ejecutar las sentencias de creación, veremos cómo **las nuevas tablas aparecen dentro de esta base de datos con ubicación personalizada**, almacenándose en la ruta que definimos previamente en el bucket.

In [0]:
USE custom;

In [0]:
%python

spark.sql("""
CREATE TABLE if not exists managed_custom
(
  width INT,
  length INT,
  height INT
)
""")

spark.sql("""
INSERT INTO managed_custom
VALUES (3, 2, 1)
""")

##################

spark.sql(f"""
CREATE TABLE external_custom
(
  width INT,
  length INT,
  height INT
)
LOCATION '{location4}/custom/'
""")

spark.sql("""
INSERT INTO external_custom
VALUES (3, 2, 1)
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

## 5.3 Metadatos de la tabla gestionada en ubicación personalizada

Si revisamos la metadata de la **tabla managed** utilizando `DESCRIBE EXTENDED`, veremos varios puntos clave.

El campo **`Type`** indica que se trata de una tabla **`MANAGED`**, lo que significa que Databricks gestiona tanto la metadata como los datos.

Sin embargo, en el campo **`Location`** podemos observar que la tabla se encuentra en la **ubicación personalizada que definimos para la base de datos**, y aparece el directorio **`custom.db`**.

Esto confirma que, aunque sea una **tabla gestionada**, sus datos se almacenan en la **ruta personalizada asociada a la base de datos**, en lugar del directorio por defecto de Unity Catalog.

In [0]:
DESCRIBE EXTENDED managed_custom

col_name,data_type,comment
width,int,null
length,int,null
height,int,null
,,
# Delta Statistics Columns,,
Column Names,"width, length, height",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,workspacedemofull02,


### 5.3.A Explicacion

---
📦 Managed Tables en Unity Catalog — ¿Dónde van los datos?

Cuando creas una **managed table** sobre un external location en S3, Unity Catalog gestiona automáticamente
el almacenamiento usando una estructura de directorios con UUIDs:

```
s3://tu-bucket/tu-path/
└── __unitystorage/
    └── schemas/
        └── {uuid-del-schema}/
            └── tables/
                └── {uuid-de-la-tabla}/
                    ├── part-00001-xxx.parquet
                    └── _delta_log/
```

#### ¿Qué es cada carpeta?

| Carpeta | Descripción |
|---|---|
| `__unitystorage/` | Directorio reservado por Unity Catalog — no modificar manualmente |
| `{uuid-del-schema}` | ID interno del schema — no el nombre visible |
| `{uuid-de-la-tabla}` | ID interno de la tabla — no el nombre visible |

#### ¿Por qué UUIDs en lugar de nombres?

- **Renombrar es gratis** → si renombras la tabla o el schema, el path en S3 no cambia
- **Sin colisiones** → dos tablas con el mismo nombre en schemas distintos nunca comparten path

#### Managed vs External

| | Managed (Unity Catalog) | External |
|---|---|---|
| Path en S3 | Gestionado por UC con UUIDs | Definido por ti con `LOCATION` |
| `DROP TABLE` | Borra los datos físicos de S3 | Solo borra el registro, los datos quedan |
| Renombrar | Sin impacto en S3 | Sin impacto en S3 |
| Acceso directo a S3 | No recomendado | Sí, es tu path |

> 💡 **Regla práctica**: con tablas managed nunca accedas directamente a la ruta `__unitystorage/`
> en S3. Siempre interactúa con los datos a través del nombre de la tabla en Databricks.

## 5.4 Metadatos de la tabla externa en la base de datos personalizada

Respecto a la segunda tabla, se trata de una **tabla externa**.

En este caso, al ejecutar `DESCRIBE EXTENDED`, veremos que el campo **`Type`** aparece como **`EXTERNAL`** y que la **ubicación (`Location`)** apunta a una ruta definida explícitamente.

Esto indica que la tabla se ha creado **fuera del directorio de la base de datos (`custom.db`)**, ya que hemos especificado manualmente su ubicación mediante la palabra clave `LOCATION`.

Por tanto, aunque la tabla esté registrada dentro de la base de datos, **sus datos no están gestionados por la ubicación de la base de datos**, sino por la ruta externa definida.

In [0]:
DESCRIBE EXTENDED external_custom

col_name,data_type,comment
width,int,null
length,int,null
height,int,null
,,
# Delta Statistics Columns,,
Column Names,"width, length, height",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,workspacedemofull02,


## 5.5 Eliminación de las tablas en Unity Catalog

Al eliminar ambas tablas, podremos comprobar que **desaparecen correctamente de Unity Catalog**.

Esto confirma que la **metadata de las tablas ha sido eliminada**, independientemente de si se trataba de una tabla **managed** o **external**.

In [0]:
DROP TABLE managed_custom;
DROP TABLE external_custom;

Revisemos en nuestro S3 las **rutas de almacenamiento** de nuestras tablas y bases de datos.

Esto nos permitirá **copiar fácilmente las rutas correspondientes**, tanto de la **base de datos managed** como de la **base de datos externa**.

De esta forma podremos navegar directamente a esos paths y comprobar el estado de los datos en el almacenamiento.

In [0]:
%python
display(dbutils.fs.ls(location4))

path,name,size,modificationTime
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/__unitystorage/,__unitystorage/,0,1776098378437
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/clustered_employees/,clustered_employees/,0,1776098378437
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/custom/,custom/,0,1776098378437
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/external_default/,external_default/,0,1776098378437
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/managed_NEW_default/,managed_NEW_default/,0,1776098378437
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/partitioned_employees/,partitioned_employees/,0,1776098378437


Verificación de la eliminación de datos en la tabla managed:

Si revisamos el almacenamiento, veremos que **ya no existe ni el directorio de datos ni los archivos asociados** a la tabla que habíamos creado como **managed** dentro de la base de datos `managed_custom`.

Esto confirma nuevamente que, en el caso de las **tablas gestionadas**, al eliminarlas se borran tanto la **metadata** como los **datos físicos en el almacenamiento**.

In [0]:
%python
display(dbutils.fs.ls("s3://mi-bucket-publico-javier-2026/tables/test_data_jm/__unitystorage/"))

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-6309019916867044>, line 1
----> 1 display(dbutils.fs.ls("s3://mi-bucket-publico-javier-2026/tables/test_data_jm/__unitystorage/"))

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:52, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     49 class ExecutionError(Exception):
     50     pass
---> 52 raise ExecutionError(str(e)) from None

ExecutionError: [RequestId=ad06122c-5709-493c-82ab-bd1d6a703e47 ErrorClass=INVALID_PARAMETER_VALUE.LOCATION_OVERLAP] Input path url 's3://mi-bucket-publico-javier-2026/tables/test_data_jm/__unitystorage' overlaps with managed storage within 'ListFiles' call. .

JVM stacktrace:
com.databricks.sql.managedcatalog.UnityCatalogServiceException
	at com.databricks.sql.managedcatalog.client.ErrorDetailsHandlerImpl.wrapServiceException(

Verificación de la tabla externa tras su eliminación

Por el contrario, si revisamos la **tabla externa**, veremos que **el directorio y los archivos siguen existiendo en el almacenamiento**.

Esto confirma nuevamente el comportamiento de las **tablas externas**: al eliminarlas, solo se borra la **metadata en Unity Catalog**, pero **los datos físicos permanecen intactos en la ubicación definida**.

In [0]:
%python
display(dbutils.fs.ls("s3://mi-bucket-publico-javier-2026/tables/test_data_jm/custom/"))

path,name,size,modificationTime
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/custom/part-00000-64e2a485-fecd-4a17-b68b-a369dd3053ca.c000.snappy.parquet,part-00000-64e2a485-fecd-4a17-b68b-a369dd3053ca.c000.snappy.parquet,977,1776097972000
s3://mi-bucket-publico-javier-2026/tables/test_data_jm/custom/_delta_log/,_delta_log/,0,1776098444638


# Summary 

# 📋 Resumen — Bases de datos y tablas en Databricks

A lo largo de este notebook hemos trabajado con las principales combinaciones de schemas y tablas en Unity Catalog. Estas son las cuatro más relevantes para el día a día:

---

## Caso 1 — Schema default + Tabla Managed

Los datos se almacenan en el storage interno de Databricks. Unity Catalog gestiona completamente la ubicación — no decides el path. Al hacer `DROP TABLE` los datos se eliminan físicamente.

**Cuándo usarlo en producción:** tablas de trabajo internas donde no necesitas controlar dónde viven los datos físicamente y el equipo opera completamente dentro de Databricks. Habitual en entornos de desarrollo o para tablas intermedias de pipelines.

---

## Caso 2 — Schema default + Tabla External

Los datos viven en la ruta de S3 que tú defines. La tabla en Databricks es solo un puntero a esa ruta. `DROP TABLE` elimina únicamente el registro — los datos en S3 permanecen intactos.

**Cuándo usarlo en producción:** datos que ya existen en S3 y son compartidos con otros sistemas externos (otros pipelines, otra plataforma, otro equipo). También cuando los datos deben sobrevivir independientemente de lo que ocurra en Databricks.

---

## Caso 3 — Schema con MANAGED LOCATION + Tabla Managed

El schema apunta a tu propio bucket de S3. La tabla managed hereda esa ubicación y Unity Catalog escribe los datos ahí usando su estructura interna de UUIDs. `DROP TABLE` elimina los datos físicamente de tu S3.

**Cuándo usarlo en producción:** cuando quieres todas las garantías de una tabla managed (permisos UC, linaje automático, auditoría, DROP limpio) pero necesitas que los datos estén en tu propia cuenta AWS — por política corporativa, por costes de almacenamiento, o para tener control sobre backups y retención.

---

## Caso 4 — Schema con MANAGED LOCATION + Tabla External

El schema tiene una ubicación base en tu S3, pero la tabla external usa su propio LOCATION independiente. `DROP TABLE` elimina solo el registro — los datos permanecen.

**Cuándo usarlo en producción:** cuando dentro de un mismo schema necesitas que ciertas tablas críticas vivan en rutas con políticas distintas — por ejemplo datos PII en un bucket encriptado separado, tablas compartidas con otros equipos que tienen su propio path en S3, o datos con requisitos de retención diferentes al resto del schema.

---